# Credit Card Fraud Detection — Training & Deployment v2
Reads featured data from EMR output, trains XGBoost, deploys endpoint, backs up to DR bucket.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q xgboost scikit-learn imbalanced-learn pyarrow s3fs pandas numpy boto3 sagemaker
!{sys.executable} -m pip install -q imbalanced-learn --upgrade

In [ ]:
# ── 2. Imports ───────────────────────────────────────────────────────────
import os, json, tarfile
import boto3
import numpy as np
import pandas as pd
import s3fs
import sagemaker
from sagemaker import get_execution_role
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, average_precision_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import xgboost as xgb

print('All imports OK')

In [ ]:
# ── 3. Config ────────────────────────────────────────────────────────────
REGION         = 'us-east-1'
PRIMARY_BUCKET = 'databucket-fraud-detection'
DR_BUCKET      = 'fraud-detection-dr'

INPUT_S3       = f's3://{PRIMARY_BUCKET}/output_data/'
ENDPOINT_NAME  = 'fraud-detection-endpoint'

# Drop columns that are not useful for training
DROP_COLS = [
    'weight', 'transaction_id', 'timestamp', 'sender_account',
    'receiver_account', 'ip_address', 'device_hash', 'ts',
    'bucket_5m'
]

sess = sagemaker.Session()
role = get_execution_role()
s3c  = boto3.client('s3', region_name=REGION)

print(f'Role : {role}')
print(f'Input: {INPUT_S3}')

In [ ]:
# ── 4. Load data from S3 (chunked to save memory) ────────────────────────
fs    = s3fs.S3FileSystem(anon=False)
files = fs.glob(f'{PRIMARY_BUCKET}/output_data/**/*.parquet')
print(f'Found {len(files)} parquet files')

dfs = []
for f in files[:10]:   # load 10 files — ~100k rows
    dfs.append(pd.read_parquet(f's3://{f}', storage_options={'anon': False}))

df = pd.concat(dfs, ignore_index=True)
print(f'Shape : {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
df.head()

In [ ]:
# ── 5. Find the fraud/target column ──────────────────────────────────────
print('All columns:')
print(df.columns.tolist())

# Auto-detect target column
candidates = ['is_fraud', 'fraud', 'Class', 'fraud_label',
               'isFraud', 'label', 'target', 'Fraud']
TARGET_COL = None
for c in candidates:
    if c in df.columns:
        TARGET_COL = c
        break

if TARGET_COL:
    print(f'\nTarget column found: "{TARGET_COL}"')
    print(df[TARGET_COL].value_counts())
    print(f'Fraud rate: {df[TARGET_COL].mean():.4%}')
else:
    print('\nTarget column NOT found! Available columns:')
    for col in df.columns:
        print(f'  {col}: {df[col].nunique()} unique values | example: {df[col].iloc[0]}')

In [ ]:
# ── 6. Prepare X / y ─────────────────────────────────────────────────────
# If TARGET_COL is None above, manually set it here:
# TARGET_COL = 'your_column_name'

assert TARGET_COL is not None, 'Set TARGET_COL manually above!'

drop = [c for c in DROP_COLS if c in df.columns]
X    = df.drop(columns=[TARGET_COL] + drop)
y    = df[TARGET_COL].astype(int)

# Encode categoricals
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    print(f'Encoding categoricals: {cat_cols}')
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Keep numeric only
X = X.select_dtypes(include=[np.number])
X = X.replace([np.inf, -np.inf], np.nan)

print(f'Features : {X.shape[1]}')
print(f'Samples  : {len(y)}')
print(f'Fraud    : {y.sum()} ({y.mean():.2%})')

In [ ]:
# ── 7. Train / test split ─────────────────────────────────────────────────
strat = y if y.nunique() == 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strat
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Train fraud: {y_train.sum()} | Test fraud: {y_test.sum()}')

In [ ]:
# ── 8. Preprocessing + SMOTE ─────────────────────────────────────────────
preprocess = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler(with_mean=False)),
])

X_train_p = preprocess.fit_transform(X_train)
X_test_p  = preprocess.transform(X_test)

try:
    sm = SMOTE(random_state=42)
    X_res, y_res = sm.fit_resample(X_train_p, y_train)
    print(f'After SMOTE — 0:{(y_res==0).sum()}  1:{(y_res==1).sum()}')
except Exception as e:
    print(f'SMOTE skipped ({e}), using original')
    X_res, y_res = X_train_p, y_train

In [ ]:
# ── 9. Train XGBoost ─────────────────────────────────────────────────────
scale_pos = max(1, int((y_res==0).sum() / max((y_res==1).sum(), 1)))

model = xgb.XGBClassifier(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos,
    eval_metric      = 'aucpr',
    random_state     = 42,
    tree_method      = 'hist',
)

model.fit(
    X_res, y_res,
    eval_set=[(X_test_p, y_test)],
    verbose=50
)
print('Training complete.')

In [ ]:
# ── 10. Evaluate ─────────────────────────────────────────────────────────
y_pred  = model.predict(X_test_p)
y_proba = model.predict_proba(X_test_p)[:, 1]

metrics = {
    'roc_auc' : round(roc_auc_score(y_test, y_proba), 4),
    'pr_auc'  : round(average_precision_score(y_test, y_proba), 4),
    'n_test'  : len(y_test),
    'n_fraud' : int(y_test.sum()),
}

print('=== Metrics ===')
print(json.dumps(metrics, indent=2))
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Legit','Fraud']))
print('\n=== Confusion Matrix ===')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# ── 11. Save & package model ─────────────────────────────────────────────
import joblib, pathlib

local_dir = pathlib.Path('/tmp/fraud_model')
local_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model,              local_dir / 'xgb_model.joblib')
joblib.dump(preprocess,         local_dir / 'preprocessor.joblib')
joblib.dump(list(X.columns),   local_dir / 'feature_names.joblib')

with open(local_dir / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

tar_path = '/tmp/model.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    for fp in local_dir.iterdir():
        tar.add(fp, arcname=fp.name)

print(f'Model packaged: {tar_path}')

In [ ]:
# ── 12. Upload to PRIMARY S3 ─────────────────────────────────────────────
def s3_upload(local_path, bucket, key):
    s3c.upload_file(local_path, bucket, key)
    print(f'Uploaded → s3://{bucket}/{key}')

s3_upload('/tmp/model.tar.gz',
          PRIMARY_BUCKET, 'models/fraud-xgb/model.tar.gz')

s3_upload(str(local_dir / 'metrics.json'),
          PRIMARY_BUCKET, 'results/fraud-xgb/metrics.json')

print('Primary upload done.')

In [ ]:
# ── 13. Backup to DR S3 ──────────────────────────────────────────────────
try:
    s3_upload('/tmp/model.tar.gz',
              DR_BUCKET, 'models/fraud-xgb/model.tar.gz')
    s3_upload(str(local_dir / 'metrics.json'),
              DR_BUCKET, 'results/fraud-xgb/metrics.json')
    print('DR backup complete.')
except Exception as e:
    print(f'DR backup failed: {e}')

In [ ]:
# ── 14. Write inference script ───────────────────────────────────────────
os.makedirs('/tmp/inference', exist_ok=True)

inference_script = '''
import joblib, os, json
import numpy as np
import pandas as pd

def model_fn(model_dir):
    return {
        'model': joblib.load(os.path.join(model_dir, 'xgb_model.joblib')),
        'prep' : joblib.load(os.path.join(model_dir, 'preprocessor.joblib')),
        'cols' : joblib.load(os.path.join(model_dir, 'feature_names.joblib')),
    }

def input_fn(body, content_type):
    if content_type == 'application/json':
        return pd.DataFrame(json.loads(body))
    raise ValueError(f'Unsupported: {content_type}')

def predict_fn(X, bundle):
    X = X.reindex(columns=bundle["cols"], fill_value=0)
    Xp = bundle["prep"].transform(X)
    return {
        'predictions'  : bundle["model"].predict(Xp).tolist(),
        'probabilities': bundle["model"].predict_proba(Xp)[:,1].tolist()
    }

def output_fn(pred, accept):
    return json.dumps(pred), 'application/json'
'''

with open('/tmp/inference/inference.py', 'w') as f:
    f.write(inference_script)

print('Inference script ready.')

In [ ]:
# ── 15. Deploy SageMaker Endpoint ────────────────────────────────────────
from sagemaker.sklearn.model import SKLearnModel

model_uri = f's3://{PRIMARY_BUCKET}/models/fraud-xgb/model.tar.gz'

sm_model = SKLearnModel(
    model_data        = model_uri,
    role              = role,
    entry_point       = '/tmp/inference/inference.py',
    framework_version = '1.2-1',
    py_version        = 'py3',
)

print(f'Deploying {ENDPOINT_NAME} ...')
predictor = sm_model.deploy(
    initial_instance_count = 1,
    instance_type          = 'ml.t2.medium',
    endpoint_name          = ENDPOINT_NAME,
)
print(f'Endpoint ready: {ENDPOINT_NAME}')

In [ ]:
# ── 16. Test endpoint ────────────────────────────────────────────────────
runtime  = boto3.client('sagemaker-runtime', region_name=REGION)
sample   = X_test.iloc[:3].to_dict(orient='records')

response = runtime.invoke_endpoint(
    EndpointName = ENDPOINT_NAME,
    ContentType  = 'application/json',
    Body         = json.dumps(sample)
)
result = json.loads(response['Body'].read())

print('Endpoint response:')
print(json.dumps(result, indent=2))
print(f'Actual labels: {y_test.iloc[:3].tolist()}')

In [ ]:
# ── 17. Summary ───────────────────────────────────────────────────────────
print('='*60)
print('DEPLOYMENT SUMMARY')
print('='*60)
print(f'Model (primary) : s3://{PRIMARY_BUCKET}/models/fraud-xgb/')
print(f'Results         : s3://{PRIMARY_BUCKET}/results/fraud-xgb/')
print(f'Model (DR)      : s3://{DR_BUCKET}/models/fraud-xgb/')
print(f'Results (DR)    : s3://{DR_BUCKET}/results/fraud-xgb/')
print(f'Endpoint        : {ENDPOINT_NAME}')
print(f'ROC-AUC         : {metrics["roc_auc"]}')
print(f'PR-AUC          : {metrics["pr_auc"]}')
print('='*60)